In [1]:
import pandas as pd
import sys
import os
import numpy as np
import scanpy as sc

In [2]:
adata = sc.read_h5ad(r"D:\Trapecar\250307_gut_liver_blood_ultimate_annotated.h5ad")

In [3]:
adata_TCR = adata[adata.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
adata_ab.obs['subject:condition']= adata_ab.obs['Donor ID'].astype(str).map(str) + ':' + adata_ab.obs['tissue+celltype'].astype(str).map(str)

C:\Users\andre\AppData\Local\Temp\ipykernel_42256\3865754011.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)


In [4]:
# from conga/tcrdist
clone_counts= pd.read_csv(r"G:\My Drive\result\publication\cellreport\revision\conga\gut_liver_TRM_clones_with0.tsv",sep = '\t',index_col = 0)
temp_dict_df = clone_counts[clone_counts['clone_size']>0][['clone_id','va_gene','vb_gene','cdr3a','cdr3b']]
temp_dict_df['clone_code'] = temp_dict_df['va_gene'].astype(str).map(str) + ' ' + temp_dict_df['vb_gene'].astype(str).map(str) + ' ' + temp_dict_df['cdr3a'].astype(str).map(str) + ' ' + temp_dict_df['cdr3b'].astype(str).map(str)
clone_replace_dict = temp_dict_df[['clone_id','clone_code']].set_index('clone_code').sort_index()['clone_id'].to_dict()

In [119]:
os.chdir(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_input")
names = ['TCRab CD4','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]
    if i == 'TCRab CD8ab':
        adata_slice = adata_slice[
            (adata_slice.obs['tissue+celltype'] != 'L TCRab CD8ab MAIT') |
            ((adata_slice.obs['tissue+celltype'] == 'L TCRab CD8ab MAIT') & (adata_slice.obs['TRAV'] == 'TRAV1-2')),
            :
        ]

    clone_df = adata_slice.obs[['subject:condition','TRAV', 'TRBV','TRBJ', 'cdr3a', 'cdr3b','clone_code']]
    clone_counts = clone_df.groupby(['subject:condition','clone_code']).size().reset_index(name='clone frequency')
    clone_counts[['TRAV','TRBV','cdr3a','cdr3b']] = clone_counts['clone_code'].str.split(' ', expand=True)
    
    Jmap = clone_df[['TRBJ','clone_code']].drop_duplicates()
    Jmap = Jmap.set_index('clone_code')
    Jmap_bdict = Jmap['TRBJ'].to_dict()
    clone_counts['TRBJ'] =clone_counts['clone_code'].replace(Jmap_bdict)

    clone_counts = clone_counts[clone_counts['clone frequency'] >= 1]
    clone_counts['clone_id'] = clone_counts['clone_code'].map(clone_replace_dict)
    clone_counts[['cdr3b','TRBV','TRBJ','cdr3a','subject:condition','clone frequency']].to_csv(i+'_GLIPH2.tsv', sep="\t",index = False, header = False)

### Mapping GLIPH2 reulst back

In [14]:
unique_clone_counts = adata_ab.obs.groupby(['Donor ID', 'tissue+celltype'])['clone_code'].nunique()
unique_clone_counts

C:\Users\andre\AppData\Local\Temp\ipykernel_42256\2367549632.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_ab.obs.groupby(['Donor ID', 'tissue+celltype'])['clone_code'].nunique()


Donor ID       tissue+celltype          
Donor AJD3280  IEL TCRab CD4 FOXP3+ Treg      86
               IEL TCRab CD4 Mobile TRM      315
               IEL TCRab CD4 TRM             500
               IEL TCRab CD8ab TRM          1102
               L TCRab CD4 FOXP3+ Treg        59
                                            ... 
Donor AJKQ118  PB TCRab CD4 Naive/TCM       2045
               PB TCRab CD4 TCM                2
               PB TCRab CD8ab Naive/TCM      343
               PB TCRab CD8ab TEM             47
               PB TCRab CD8ab Teff            69
Name: clone_code, Length: 87, dtype: int64

In [15]:
unique_clone_counts_table = unique_clone_counts.unstack(fill_value=0)
unique_clone_counts_table

tissue+celltype,IEL TCRab CD4 FOXP3+ Treg,IEL TCRab CD4 Mobile TRM,IEL TCRab CD4 TRM,IEL TCRab CD8ab TRM,L TCRab CD4 FOXP3+ Treg,L TCRab CD4 Naive/TCM,L TCRab CD4 TCM,L TCRab CD4 TRM,L TCRab CD8aa MAIT,L TCRab CD8ab MAIT,...,LP TCRab CD8aa MAIT,LP TCRab CD8ab TCM,LP TCRab CD8ab TEM,LP TCRab CD8ab TRM,PB TCRab CD4 FOXP3+ Treg,PB TCRab CD4 Naive/TCM,PB TCRab CD4 TCM,PB TCRab CD8ab Naive/TCM,PB TCRab CD8ab TEM,PB TCRab CD8ab Teff
Donor ID,,,,,,,,,,,,,,,,,,,,,
Donor AJD3280,86,315,500,1102,59,255,211,689,449,97,...,38,29,68,149,205,1568,5,582,50,164
Donor AJG2309,9,44,53,730,33,34,61,157,95,49,...,2,13,207,283,221,1128,26,86,60,167
Donor AJKQ118,25,35,113,404,13,45,57,429,165,58,...,2,12,82,42,499,2045,2,343,47,69


In [ ]:
# CD4_mask =['CD4' in i for i in unique_clone_counts_table.columns]
# CD8_mask =['CD8' in i for i in unique_clone_counts_table.columns]

#### CD4

In [11]:
CD4_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD4_GLIPH2.csv",index_col = 0).iloc[:,0:18]

In [88]:
CD4_GLIPH2['celltype'] = CD4_GLIPH2['Sample'].str.split(':').str[1]
CD4_GLIPH2['donor']    = CD4_GLIPH2['Sample'].str.split(':').str[0]
CD4_GLIPH2['clone'] = (
    CD4_GLIPH2
    .groupby(['donor','TcRb','V','J','TcRa'], dropna=False, sort=False)
    .ngroup()
)
motif_col = 'type'
dedup = (
    CD4_GLIPH2
    .loc[:, ['donor','celltype',motif_col,'clone']]
    .drop_duplicates()
)

In [102]:
CD4_GLIPH2_donor_clone_normalized = {}

for i in dedup['donor'].unique():
    df_i = dedup[dedup['donor'] == i]

    nodes = sorted(df_i['celltype'].unique())
    shared_counts = pd.DataFrame(0.0, index=nodes, columns=nodes, dtype=float)

    # Precompute sets for speed
    clones_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, 'clone']) for ct in nodes}
    motifs_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, motif_col]) for ct in nodes}

    for A in nodes:
        denom = unique_clone_counts_table.loc[i,A]
        if denom == 0:
            continue
        for B in nodes:
            if A == B:
                continue
            # Motifs present in B
            motifs_B = motifs_in_ct[B]
            if not motifs_B:
                shared_counts.loc[A, B] = 0.0
                continue

            # Clones in A that have at least one motif also seen in B
            clones_A_shared = set(
                df_i.loc[
                    (df_i['celltype']==A) & (df_i[motif_col].isin(motifs_B)), #well that is, if a clone in cell type A and has the motif that is in cell type B
                    'clone'
                ]
            )
            shared_counts.loc[A, B] = len(clones_A_shared) / denom

    CD4_GLIPH2_donor_clone_normalized[i] = shared_counts

# --- 2) Average across donors (aligning all cell types) ---
all_nodes = sorted(dedup['celltype'].unique())
dfs = [m.reindex(index=all_nodes, columns=all_nodes, fill_value=0.0)
       for m in CD4_GLIPH2_donor_clone_normalized.values()]
CD4_GLIPH2_donor_clone_normalized_mean = sum(dfs) / len(dfs)
CD4_GLIPH2_donor_clone_normalized_mean

,IEL TCRab CD4 FOXP3+ Treg,IEL TCRab CD4 Mobile TRM,IEL TCRab CD4 TRM,L TCRab CD4 FOXP3+ Treg,L TCRab CD4 Naive/TCM,L TCRab CD4 TCM,L TCRab CD4 TRM,LP TCRab CD4 FOXP3+ Treg,LP TCRab CD4 Mobile TRM,LP TCRab CD4 Naive/TCM,LP TCRab CD4 Poised TCM,LP TCRab CD4 TRM,LP TCRab CD4 Tph,PB TCRab CD4 FOXP3+ Treg,PB TCRab CD4 Naive/TCM,PB TCRab CD4 TCM
IEL TCRab CD4 FOXP3+ Treg,0.000000,0.000000,0.015504,0.003876,0.000000,0.000000,0.017209,0.218725,0.000000,0.000000,0.028837,0.068837,0.007752,0.070543,0.064961,0.037037
IEL TCRab CD4 Mobile TRM,0.000000,0.000000,0.027682,0.002116,0.001058,0.002116,0.060317,0.002116,0.077417,0.000000,0.006349,0.092232,0.008466,0.029630,0.031746,0.000000
IEL TCRab CD4 TRM,0.002667,0.015906,0.000000,0.000000,0.024089,0.018749,0.094702,0.022868,0.009617,0.000667,0.006283,0.221983,0.041799,0.019806,0.053505,0.000000
L TCRab CD4 FOXP3+ Treg,0.005650,0.011299,0.000000,0.000000,0.000000,0.000000,0.041392,0.011299,0.005650,0.000000,0.005650,0.011299,0.000000,0.040404,0.043999,0.000000
L TCRab CD4 Naive/TCM,0.000000,0.001307,0.047277,0.000000,0.000000,0.022222,0.133551,0.001307,0.003922,0.000000,0.003922,0.059695,0.012636,0.007407,0.045969,0.000000
L TCRab CD4 TCM,0.000000,0.003160,0.038718,0.000000,0.017544,0.000000,0.165456,0.004739,0.000000,0.001580,0.008624,0.061814,0.006319,0.050418,0.093239,0.000000
L TCRab CD4 TRM,0.001261,0.005336,0.034713,0.003384,0.017267,0.033427,0.000000,0.004058,0.024863,0.002607,0.003384,0.112578,0.005043,0.045645,0.062385,0.000000
LP TCRab CD4 FOXP3+ Treg,0.053779,0.002743,0.012106,0.002743,0.001372,0.004115,0.007425,0.000000,0.008557,0.000000,0.001372,0.032397,0.007425,0.015609,0.038711,0.001938
LP TCRab CD4 Mobile TRM,0.000000,0.035128,0.017190,0.001297,0.003891,0.000000,0.061967,0.007696,0.000000,0.000000,0.005995,0.098504,0.008186,0.015252,0.040380,0.000000
LP TCRab CD4 Naive/TCM,0.000000,0.000000,0.006667,0.000000,0.000000,0.006667,0.013469,0.000000,0.000000,0.000000,0.000000,0.047908,0.000000,0.048469,0.040833,0.000000


In [ ]:
# CD4_GLIPH2_donor_motif_normalized_mean[CD4_GLIPH2_donor_motif_normalized_mean<np.percentile(CD4_GLIPH2_donor_motif_normalized_mean.values.flatten(),90)] = 0

In [103]:
CD4_GLIPH2_donor_motif_normalized_mean.to_csv("G:/My Drive/result/publication/cellreport/revision/GLIPH2/GLIPH2_results/CD4_GLIPH2_donor_separated_motif_shared_normalized_mean.csv")

#### now do the same for CD8ab

In [120]:
CD8_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD8_GLIPH2_2.csv",index_col = 0).iloc[:,0:18]
CD8_GLIPH2['celltype'] = CD8_GLIPH2['Sample'].str.split(':').str[1]
CD8_GLIPH2['donor']    = CD8_GLIPH2['Sample'].str.split(':').str[0]
CD8_GLIPH2['clone'] = (
    CD8_GLIPH2
    .groupby(['donor','TcRb','V','J','TcRa'], dropna=False, sort=False)
    .ngroup()
)
motif_col = 'type'
dedup = (
    CD8_GLIPH2
    .loc[:, ['donor','celltype',motif_col,'clone']]
    .drop_duplicates()
)
CD8_GLIPH2_donor_clone_normalized = {}

for i in dedup['donor'].unique():
    df_i = dedup[dedup['donor'] == i]

    nodes = sorted(df_i['celltype'].unique())
    shared_counts = pd.DataFrame(0.0, index=nodes, columns=nodes, dtype=float)

    # Precompute sets for speed
    clones_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, 'clone']) for ct in nodes}
    motifs_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, motif_col]) for ct in nodes}

    for A in nodes:
        denom = unique_clone_counts_table.loc[i,A]
        if denom == 0:
            continue
        for B in nodes:
            if A == B:
                continue
            # Motifs present in B
            motifs_B = motifs_in_ct[B]
            if not motifs_B:
                shared_counts.loc[A, B] = 0.0
                continue

            # Clones in A that have at least one motif also seen in B
            clones_A_shared = set(
                df_i.loc[
                    (df_i['celltype']==A) & (df_i[motif_col].isin(motifs_B)), #well that is, if a clone in cell type A and has the motif that is in cell type B
                    'clone'
                ]
            )
            shared_counts.loc[A, B] = len(clones_A_shared) / denom

    CD8_GLIPH2_donor_clone_normalized[i] = shared_counts

# --- 2) Average across donors (aligning all cell types) ---
all_nodes = sorted(dedup['celltype'].unique())
dfs = [m.reindex(index=all_nodes, columns=all_nodes, fill_value=0.0)
       for m in CD8_GLIPH2_donor_clone_normalized.values()]
CD8_GLIPH2_donor_clone_normalized_mean = sum(dfs) / len(dfs)
CD8_GLIPH2_donor_clone_normalized_mean
# CD8_GLIPH2_donor_motif_normalized_mean[CD8_GLIPH2_donor_motif_normalized_mean<np.percentile(CD8_GLIPH2_donor_motif_normalized_mean.values.flatten(),90)] = 0
CD8_GLIPH2_donor_clone_normalized_mean.to_csv("G:/My Drive/result/publication/cellreport/revision/GLIPH2/GLIPH2_results/CD8_GLIPH2_donor_separated_motif_shared_normalized_mean.csv")

In [ ]:
CD8_GLIPH2_donor_clone_normalized_mean

,IEL TCRab CD8ab TRM,L TCRab CD8ab MAIT,L TCRab CD8ab Naive/TCM,L TCRab CD8ab TRM,L TCRab CD8ab Teff,LP TCRab CD8ab TCM,LP TCRab CD8ab TEM,LP TCRab CD8ab TRM,PB TCRab CD8ab Naive/TCM,PB TCRab CD8ab TEM,PB TCRab CD8ab Teff
IEL TCRab CD8ab TRM,0.000000,0.003163,0.016644,0.069901,0.019351,0.004450,0.055651,0.098228,0.019927,0.011029,0.023416
L TCRab CD8ab MAIT,0.022929,0.000000,0.017042,0.110940,0.022789,0.000000,0.000000,0.000000,0.012620,0.000000,0.017042
L TCRab CD8ab Naive/TCM,0.146448,0.016098,0.000000,0.240786,0.043100,0.017045,0.063447,0.049268,0.047835,0.030303,0.105626
L TCRab CD8ab TRM,0.107930,0.021570,0.036344,0.000000,0.078563,0.011388,0.069151,0.030276,0.024326,0.048886,0.130037
L TCRab CD8ab Teff,0.152437,0.020013,0.033268,0.315660,0.000000,0.022482,0.164392,0.040936,0.022482,0.093307,0.453411
LP TCRab CD8ab TCM,0.183245,0.000000,0.048630,0.151194,0.090554,0.000000,0.181624,0.141836,0.037135,0.183761,0.204612
LP TCRab CD8ab TEM,0.248580,0.000000,0.024368,0.119832,0.099900,0.016947,0.000000,0.110944,0.008052,0.059115,0.129643
LP TCRab CD8ab TRM,0.445786,0.000000,0.020466,0.086575,0.015004,0.014885,0.118069,0.000000,0.043050,0.027533,0.027604
PB TCRab CD8ab Naive/TCM,0.050382,0.001545,0.015249,0.031470,0.007538,0.004449,0.011628,0.016846,0.000000,0.012559,0.013132
PB TCRab CD8ab TEM,0.114090,0.000000,0.011111,0.179480,0.111182,0.049054,0.133664,0.065296,0.045296,0.000000,0.193830
